# 【notebook｜N-06】训练工程：AMP / 梯度累积 / 断点续训
> 对应：《05-训练循环与数据管线.md》【文档｜DOC-5】；代码 `F-b01-train`、`F-b02-pretrain`

> **对应理论篇**：[`05-训练循环与数据管线.md`](../../01-基础/05-训练循环与数据管线.md)
> **对应 AlphaProof 源码**：`nanoproof/nanoproof/common.py:38` `_detect_compute_dtype`、`common.py:86` `get_lr_multiplier`、`nanoproof/checkpoints.py`、`pretrain.py`
> **Colab**：CPU 可完整跑通；GPU 段落自动启用。

## 学习目标
1. 理解梯度累积为何等价于大 batch，并数值验证；
2. 理解 AMP（bf16/fp16）与 GradScaler 的作用与适用条件；
3. 掌握 checkpoint 保存/恢复，保证续训与不间断训练逐位一致。


In [1]:
# 【N-06.c1】
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu 2>/dev/null || pip -q install torch

## 理论回顾

梯度累积把大 batch 梯度拆成 $K$ 个 micro-batch：

$$
g=\frac1K\sum_{k=1}^{K}\nabla_\theta\ell(\mathcal B_k)
$$

AMP：低精度做矩阵乘，master weight 保持 fp32。bf16 动态范围同 fp32（一般不需
GradScaler），fp16 需 GradScaler 防下溢。续训需保存模型、优化器状态（Adam 的 $m,v$）、
调度进度、（可选）数据位置。


## 实验 1：梯度累积 == 大 batch（数值验证）

In [2]:
# 【N-06.c2】
import torch, torch.nn as nn, copy

torch.manual_seed(0)
X = torch.randn(16, 8); Y = torch.randn(16, 1)

def make_model():
    torch.manual_seed(123)
    return nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 1))

A = make_model(); B = copy.deepcopy(A)
# 一次大 batch(16)
loss = nn.functional.mse_loss(A(X), Y); A.zero_grad(); loss.backward()
g_big = {n: p.grad.clone() for n, p in A.named_parameters()}
# 两个 micro-batch(8)，loss/2 累积
B.zero_grad()
for i in range(2):
    sl = slice(i * 8, (i + 1) * 8)
    (nn.functional.mse_loss(B(X[sl]), Y[sl]) / 2).backward()
for n, p in B.named_parameters():
    print(f"{n:12s} max|big-accum| = {(g_big[n] - p.grad).abs().max().item():.2e}")


0.weight     max|big-accum| = 7.45e-09
0.bias       max|big-accum| = 7.45e-09
2.weight     max|big-accum| = 7.45e-09
2.bias       max|big-accum| = 2.98e-08


## 实验 2：AMP（GPU 才有加速；CPU 演示等价性）

In [3]:
# 【N-06.c3】
import time, torch, torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)
Xg = torch.randn(1024, 256, device=device); Yg = torch.randn(1024, 1, device=device)
model = nn.Sequential(nn.Linear(256, 512), nn.GELU(), nn.Linear(512, 1)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

def one_step(use_amp):
    model.zero_grad(set_to_none=True)
    if use_amp and device == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            loss = nn.functional.mse_loss(model(Xg), Yg)
    else:
        loss = nn.functional.mse_loss(model(Xg), Yg)
    loss.backward(); opt.step()
    return loss.item()

for tag, amp in [("fp32", False), ("amp", True)]:
    if amp and device != "cuda":
        print("amp: 跳过（无 GPU）；CUDA 上会用 bfloat16 autocast")
        continue
    t0 = time.time()
    for _ in range(20):
        l = one_step(amp)
    print(f"{tag:5s} loss {l:.4f} 20 步 {time.time()-t0:.2f}s")


device = cuda


fp32  loss 0.1591 20 步 1.94s


amp   loss 0.0156 20 步 2.51s


## 实验 3：checkpoint 续训 vs 不间断训练

In [4]:
# 【N-06.c4】
import torch, torch.nn as nn

def build():
    torch.manual_seed(7)
    m = nn.Sequential(nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 1))
    return m, torch.optim.AdamW(m.parameters(), lr=1e-2)

torch.manual_seed(0)
X = torch.randn(32, 4); Y = torch.randn(32, 1)

def train_steps(m, opt, n):
    for _ in range(n):
        loss = nn.functional.mse_loss(m(X), Y)
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

mA, oA = build(); lossA = train_steps(mA, oA, 30)          # 一口气 30 步
mB, oB = build(); train_steps(mB, oB, 15)                   # 前半
ckpt = {"model": mB.state_dict(), "optim": oB.state_dict()}
mC, oC = build(); mC.load_state_dict(ckpt["model"]); oC.load_state_dict(ckpt["optim"])
lossB = train_steps(mC, oC, 15)                             # 后半
same = all(torch.equal(a, c) for a, c in zip(mA.parameters(), mC.parameters()))
print("loss 不间断:", round(lossA, 6), "| loss 续训:", round(lossB, 6))
print("恢复后参数完全一致:", same)


loss 不间断: 0.479996 | loss 续训: 0.479996
恢复后参数完全一致: True


## 思考题

1. 梯度累积时为什么 loss 要除以累积步数 $K$？不除会怎样？
2. 为什么 bf16 训练通常不需要 GradScaler，而 fp16 需要？
3. 续训若只保存模型参数、不保存优化器状态，会出什么问题？
4. nanoproof 用自定义 `Linear` 显式 dtype 取代 autocast（`model.py:49`），好处是什么？

## 改动实验（任选 2–3 个）
1. 把累积步数改成 4（micro-batch=4），再验证与大 batch=16 的梯度差。
2. 在实验 3 去掉优化器状态恢复，观察续训 loss 是否仍与不间断一致。
3. GPU 上把 `bfloat16` 换成 `float16` 并加 `torch.amp.GradScaler`，比较稳定性。
4. 用 `train.py --resume out/micro/last.pt` 实际跑一次断点续训。

## 预期输出
```
0.weight     max|big-accum| = 0.00e+00  （所有层都应 ~1e-8 以下）
device = cuda/cpu
fp32 loss ... （cuda 上 amp 更快，loss 与 fp32 接近）
loss 不间断: 0.xxxxxx | loss 续训: 0.xxxxxx
恢复后参数完全一致: True
```
